In [ ]:
import os
import re
import json
import joblib
import requests
import numpy as np
import pandas as pd
from jsonschema import validate, ValidationError

os.environ['LLM_API_KEY'] = ""

LLM_URL = "https://openrouter.ai/api/v1/chat/completions"
LLM_MODEL = "openai/gpt-3.5-turbo"  
API_KEY = os.environ.get("LLM_API_KEY")

print("API key loaded:", bool(API_KEY and API_KEY != "PASTE_YOUR_OPENROUTER_KEY_HERE"))

In [ ]:
import getpass
os.environ['LLM_API_KEY'] = getpass.getpass("Paste your OpenRouter key: ")

### call_llm function

In [ ]:
def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):
    if not API_KEY:
        print("LLM call skipped: LLM_API_KEY environment variable is not set.")
        return None

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": LLM_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    response = requests.post(LLM_URL, headers=headers, json=payload)

    if response.status_code != 200:
        print(f"LLM call failed. Status code: {response.status_code}")
        print(response.text[:500])
        return None

    return response.json()["choices"][0]["message"]["content"]

### SMOKE TEST

In [ ]:

test_reply = call_llm(
    system_prompt="You are a terse assistant.",
    user_prompt="Reply with only the word: hello",
    temperature=0.0,
    max_tokens=10,
)
print(f"call_llm test reply: {test_reply!r}")

### PII guardrail

In [ ]:
def has_pii(text):
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'
    return bool(re.search(email_pattern, text) or re.search(phone_pattern, text))

def guarded_call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):
    if has_pii(user_prompt):
        print("Input blocked: PII detected.")
        return None
    return call_llm(system_prompt, user_prompt, temperature=temperature, max_tokens=max_tokens)

### Guardrail demonstration

In [ ]:
pii_input = "Predicted class: 1. Contact john.doe@example.com for questions."
clean_input = "Predicted class: 1. Probability: 0.87. Features: Gr_Liv_Area=2100, Overall_Qual=7."

print("\n--- Test 1: input WITH email (should be blocked) ---")
result_blocked = guarded_call_llm("You are an assistant.", pii_input)
print(f"Result: {result_blocked}")

print("\n--- Test 2: input WITHOUT PII (should proceed) ---")
result_allowed = guarded_call_llm("You are an assistant.", clean_input)
print(f"Result (truncated): {str(result_allowed)[:200]}")

### Load model + rebuild feature space + encode_record

In [ ]:

best_model = joblib.load("best_model.pkl")
print("best_model.pkl loaded successfully.")

df = pd.read_csv("cleaned_data.csv")
TARGET_REG = "SalePrice"
y_reg_full = df[TARGET_REG].astype(float)
X_full = df.drop(columns=[TARGET_REG])

QUALITY_MAP = {"Po": 0, "Fa": 1, "TA": 2, "Gd": 3, "Ex": 4}
ORDINAL_QUALITY_COLS = [
    "Exter_Qual", "Exter_Cond", "Bsmt_Qual", "Bsmt_Cond",
    "Heating_QC", "Kitchen_Qual", "Garage_Qual", "Garage_Cond",
]
for col in ORDINAL_QUALITY_COLS:
    X_full[col] = X_full[col].map(QUALITY_MAP)

ORDINAL_OTHER_MAPS = {
    "Lot_Shape":     {"Reg": 3, "IR1": 2, "IR2": 1, "IR3": 0},
    "Land_Slope":    {"Gtl": 2, "Mod": 1, "Sev": 0},
    "Bsmt_Exposure": {"Gd": 3, "Av": 2, "Mn": 1, "No": 0},
    "BsmtFin_Type_1":{"GLQ": 5, "ALQ": 4, "BLQ": 3, "Rec": 2, "LwQ": 1, "Unf": 0},
    "BsmtFin_Type_2":{"GLQ": 5, "ALQ": 4, "BLQ": 3, "Rec": 2, "LwQ": 1, "Unf": 0},
    "Garage_Finish": {"Fin": 2, "RFn": 1, "Unf": 0},
    "Paved_Drive":   {"Y": 2, "P": 1, "N": 0},
    "Utilities":     {"AllPub": 2, "NoSewr": 1, "NoSeWa": 0},
    "Functional":    {"Typ": 7, "Min1": 6, "Min2": 5, "Mod": 4, "Maj1": 3, "Maj2": 2, "Sev": 1, "Sal": 0},
    "Central_Air":   {"Y": 1, "N": 0},
}
for col, mapping in ORDINAL_OTHER_MAPS.items():
    X_full[col] = X_full[col].map(mapping)

nominal_cols = [c for c in X_full.columns if X_full[c].dtype == object or str(X_full[c].dtype).lower() == "str"]
X_full = pd.get_dummies(X_full, columns=nominal_cols, drop_first=True)
X_full = X_full.fillna(X_full.median(numeric_only=True)).astype(float)

FEATURE_NAMES = X_full.columns.tolist()
FEATURE_MEDIANS = X_full.median(numeric_only=True)

print(f"Encoded feature space has {len(FEATURE_NAMES)} columns.")

def encode_record(features: dict) -> pd.DataFrame:
    """
    Takes {feature_name: value} in the ALREADY-ENCODED feature space
    (same columns best_model.pkl was trained on). Missing features are
    filled with that column's training-set median.
    """
    row = FEATURE_MEDIANS.copy()
    for key, value in features.items():
        if key in row.index:
            row[key] = value
        else:
            print(f"  (warning) '{key}' not a recognized feature — ignored.")
    return pd.DataFrame([row])[FEATURE_NAMES]

### Hand-crafted inputs + JSON schema + prompts

In [ ]:
hand_crafted_inputs = [
    {  # Large, high-quality home -> expect predicted class 1
        "Gr_Liv_Area": 2600, "Overall_Qual": 9, "Overall_Cond": 8,
        "Garage_Cars": 3, "Total_Bsmt_SF": 1800, "Year_Built": 2015,
    },
    {  # Small, dated, low-quality home -> expect predicted class 0
        "Gr_Liv_Area": 800, "Overall_Qual": 3, "Overall_Cond": 4,
        "Garage_Cars": 0, "Total_Bsmt_SF": 500, "Year_Built": 1930,
    },
    {  # Mid-range home -> borderline case
        "Gr_Liv_Area": 1450, "Overall_Qual": 6, "Overall_Cond": 6,
        "Garage_Cars": 2, "Total_Bsmt_SF": 1000, "Year_Built": 1995,
    },
]

EXPLANATION_SCHEMA = {
    "type": "object",
    "properties": {
        "prediction_label": {"type": "string"},
        "confidence_level": {"type": "string", "enum": ["low", "medium", "high"]},
        "top_reason": {"type": "string"},
        "second_reason": {"type": "string"},
        "next_step": {"type": "string"},
    },
    "required": [
        "prediction_label", "confidence_level",
        "top_reason", "second_reason", "next_step",
    ],
}

FALLBACK_EXPLANATION = {
    "prediction_label": None, "confidence_level": None,
    "top_reason": None, "second_reason": None, "next_step": None,
}

SYSTEM_PROMPT_C = (
    "You are a housing-price model explainer. You will be given a home's "
    "feature values, the model's predicted class (0 = below-median price, "
    "1 = above-median price), and the model's predicted probability for "
    "that class. Respond with ONLY a valid JSON object (no markdown, no "
    "commentary) with exactly these fields: "
    '{"prediction_label": string, "confidence_level": "low"|"medium"|"high", '
    '"top_reason": string, "second_reason": string, "next_step": string}. '
    "top_reason and second_reason must reference specific feature values "
    "provided to you. next_step should be a short actionable recommendation "
    "for a real-estate agent."
)

USER_PROMPT_TEMPLATE_C = (
    "Feature values: {features}\n"
    "Predicted class: {pred_class}\n"
    "Predicted probability (of predicted class): {pred_proba:.4f}\n"
    "Explain this prediction as instructed."
)

def parse_and_validate(raw_response, schema, fallback):
    if raw_response is None:
        return dict(fallback), "fail (no response / blocked)"
    try:
        parsed = json.loads(raw_response.strip())
    except json.JSONDecodeError as e:
        print(f"  JSON decode error: {e}")
        return dict(fallback), f"fail (JSONDecodeError: {e})"
    try:
        validate(instance=parsed, schema=schema)
    except ValidationError as e:
        print(f"  Schema validation error: {e.message}")
        return dict(fallback), f"fail (ValidationError: {e.message})"
    return parsed, "pass"

###  Temperature A/B comparison

In [ ]:


ab_rows = []
for i, features in enumerate(hand_crafted_inputs):
    encoded = encode_record(features)
    pred_class = int(best_model.predict(encoded)[0])
    pred_proba = float(best_model.predict_proba(encoded)[0][pred_class])

    user_prompt = USER_PROMPT_TEMPLATE_C.format(
        features=features, pred_class=pred_class, pred_proba=pred_proba
    )

    out_temp0 = guarded_call_llm(SYSTEM_PROMPT_C, user_prompt, temperature=0.0)
    out_temp07 = guarded_call_llm(SYSTEM_PROMPT_C, user_prompt, temperature=0.7)

    ab_rows.append({"Input": f"Record {i+1}", "Output_temp0": out_temp0, "Output_temp0.7": out_temp07})
    print(f"\nRecord {i+1}:")
    print(f"  temp=0.0  -> {str(out_temp0)[:200]}")
    print(f"  temp=0.7  -> {str(out_temp07)[:200]}")

ab_df = pd.DataFrame(ab_rows)
print("\nTemperature A/B table:")
print(ab_df.to_string(index=False))

### End-to-end demonstration 

In [ ]:


demo_rows = []
for i, features in enumerate(hand_crafted_inputs):
    print(f"\n--- Input {i+1}: {features} ---")

    encoded = encode_record(features)
    pred_class = int(best_model.predict(encoded)[0])
    pred_proba = float(best_model.predict_proba(encoded)[0][pred_class])
    print(f"Predicted class: {pred_class}, Predicted probability: {pred_proba:.4f}")

    user_prompt = USER_PROMPT_TEMPLATE_C.format(
        features=features, pred_class=pred_class, pred_proba=pred_proba
    )

    raw_response = guarded_call_llm(SYSTEM_PROMPT_C, user_prompt, temperature=0.0)
    print(f"Raw LLM response: {raw_response}")

    parsed, status = parse_and_validate(raw_response, EXPLANATION_SCHEMA, FALLBACK_EXPLANATION)
    print(f"Validation outcome: {status}")
    print(f"Parsed explanation: {parsed}")

    demo_rows.append({
        "Feature_Input": features, "Predicted_Class": pred_class,
        "Probability": round(pred_proba, 4), "Explanation_JSON": parsed,
        "Validation_Status": status,
    })

demo_df = pd.DataFrame(demo_rows)
print("\n=== FINAL 3-ROW DEMONSTRATION TABLE (paste into README) ===")
print(demo_df.to_string(index=False))

os.makedirs("outputs", exist_ok=True)
demo_df.to_csv("outputs/part4_demo_table.csv", index=False)
ab_df.to_csv("outputs/part4_temperature_ab.csv", index=False)
print("\nSaved outputs/part4_demo_table.csv and outputs/part4_temperature_ab.csv")